# 🐶 Dogdex Training Notebook (Colab Version)

Este notebook foi otimizado para o Google Colab. Ele utiliza **EfficientNetB0** para maior precisão e sincroniza automaticamente as raças com o backend através do arquivo `labels.json`.

## 📦 1. Preparar Ambiente

In [ ]:
!pip install tensorflowjs kaggle
import os
import json
import tensorflow as tf
from tensorflow.keras import layers, models
from google.colab import files

## 🔑 2. Configurar Kaggle
Faça o upload do seu arquivo `kaggle.json` para baixar o dataset automaticamente.

In [ ]:
if not os.path.exists('/root/.kaggle/kaggle.json'):
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d jessicali9530/stanford-dogs-dataset
!unzip -q stanford-dogs-dataset.zip -d ./dataset

## 📁 3. Carregar Dataset

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = './dataset/images/Images'

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
print(f"Total de raças encontradas: {len(class_names)}")

# Salvar labels.json para o backend
with open('labels.json', 'w') as f:
    json.dump(class_names, f)

print("✅ labels.json gerado!")

## ⚙️ 4. Configurar Modelo (EfficientNetB0)
Nota: Data Augmentation é usada apenas no treinamento para facilitar a exportação para TFJS.

In [ ]:
# Base model
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, 
    weights='imagenet', 
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 🏗️ 5. Treinamento

In [ ]:
# Definimos a augmentation aqui como um wrapper para o treinamento
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.2),
])

def augment(image, label):
  return data_augmentation(image), label

train_ds_aug = train_ds.map(augment)

print("--- Inciando Fase 1 (Top layers) ---")
history = model.fit(
    train_ds_aug,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
print("--- Iniciando Fase 2 (Fine-tuning) ---")
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5), 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

history_fine = model.fit(
    train_ds_aug,
    validation_data=val_ds,
    epochs=10
)

## 📁 6. Exportar para TFJS (GraphModel)
Exportaremos como GraphModel, que é mais estável no Node.js.

In [ ]:
# Exportar como SavedModel primeiro
model.export('dogdex_model_saved')

# Converter para TFJS GraphModel
!mkdir -p tfjs_model
!tensorflowjs_converter --input_format tf_saved_model dogdex_model_saved/ tfjs_model/

!zip -r dogdex_model.zip tfjs_model/ labels.json

print("✅ Exportação Concluída!")
files.download('dogdex_model.zip')